In [1]:
import os
from pathlib import Path
import json

import numpy as np
import pandas as pd
import xarray as xr

from tensorflow import keras

PROJECT_DIR = Path(r"C:\Users\Student\Desktop\SIS")


SARAH_FILE   = PROJECT_DIR / "data" / "SARAH3" / "sarah3_sis_eindhoven_0.05_2024-01-01_2025-04-01.nc"
CAMS_EAC4    = PROJECT_DIR / "data" / "CAMS"   / "cams_eac4_ehv_0.05_daily.nc"
CAMS_NRT     = PROJECT_DIR / "data" / "CAMS"   / "cams_nrt_ehv_0.05_daily.nc"
DEM_FILE     = PROJECT_DIR / "data" / "DEM"    / "dem_slope_aspect_on_sarah_0.05_ehv.nc"
S2_SARAH_NC  = PROJECT_DIR / "data" / "S2_On_SARAH" / "s2_monthly_features_on_sarah_0.05_2024.nc"


MODEL_A_PATH = PROJECT_DIR / "models" / "unet_modelA_Kt_huber_medium_best.keras"
MODEL_B_PATH = PROJECT_DIR / "models" / "unet_modelB_Kt_huber_medium_best.keras"

A_MEAN_PATH = PROJECT_DIR / "models" / "unetA_train_mean.npy"
A_STD_PATH  = PROJECT_DIR / "models" / "unetA_train_std.npy"
B_MEAN_PATH = PROJECT_DIR / "models" / "unetB_train_mean.npy"
B_STD_PATH  = PROJECT_DIR / "models" / "unetB_train_std.npy"


OUT_PRE = PROJECT_DIR / "data" / "precomputed"
OUT_EXP = PROJECT_DIR / "data" / "explain"
OUT_PRE.mkdir(parents=True, exist_ok=True)
OUT_EXP.mkdir(parents=True, exist_ok=True)

print("✅ Output folders ready:")
print(" -", OUT_PRE)
print(" -", OUT_EXP)

# Quick existence checks
for p in [SARAH_FILE, CAMS_EAC4, CAMS_NRT, DEM_FILE, S2_SARAH_NC, MODEL_A_PATH, MODEL_B_PATH, A_MEAN_PATH, A_STD_PATH, B_MEAN_PATH, B_STD_PATH]:
    print(f"{p.name:45s}", "✅" if p.exists() else "❌ MISSING")


✅ Output folders ready:
 - C:\Users\Student\Desktop\SIS\data\precomputed
 - C:\Users\Student\Desktop\SIS\data\explain
sarah3_sis_eindhoven_0.05_2024-01-01_2025-04-01.nc ✅
cams_eac4_ehv_0.05_daily.nc                   ✅
cams_nrt_ehv_0.05_daily.nc                    ✅
dem_slope_aspect_on_sarah_0.05_ehv.nc         ✅
s2_monthly_features_on_sarah_0.05_2024.nc     ✅
unet_modelA_Kt_huber_medium_best.keras        ✅
unet_modelB_Kt_huber_medium_best.keras        ✅
unetA_train_mean.npy                          ✅
unetA_train_std.npy                           ✅
unetB_train_mean.npy                          ✅
unetB_train_std.npy                           ✅


In [2]:
# =========================
# 2.1 Load datasets
# =========================
sarah = xr.open_dataset(SARAH_FILE)
cams_eac4 = xr.open_dataset(CAMS_EAC4)
cams_nrt  = xr.open_dataset(CAMS_NRT)
dem = xr.open_dataset(DEM_FILE)
s2_monthly = xr.open_dataset(S2_SARAH_NC)

print("SARAH:", sarah.sizes, " | vars:", list(sarah.data_vars))
print("CAMS EAC4:", cams_eac4.sizes, " | vars:", list(cams_eac4.data_vars))
print("CAMS NRT :", cams_nrt.sizes, " | vars:", list(cams_nrt.data_vars))
print("DEM:", dem.sizes, " | vars:", list(dem.data_vars))
print("S2 monthly:", s2_monthly.sizes, " | vars:", list(s2_monthly.data_vars))

SARAH: Frozen({'time': 457, 'lat': 20, 'lon': 20})  | vars: ['SIS']
CAMS EAC4: Frozen({'time': 366, 'lat': 20, 'lon': 20})  | vars: ['spatial_ref', 'tcc', 'aod550', 'tcwv']
CAMS NRT : Frozen({'time': 90, 'lat': 20, 'lon': 20})  | vars: ['spatial_ref', 'tcwv', 'aod550']
DEM: Frozen({'x': 20, 'y': 20})  | vars: ['spatial_ref', 'elevation', 'slope', 'aspect']
S2 monthly: Frozen({'month': 12, 'lat': 20, 'lon': 20})  | vars: ['ndvi', 'brightness', 'albedo', 'nir']


In [3]:
# =========================
# 2.2 Combine CAMS + align
# =========================
cams_all = xr.concat([cams_eac4, cams_nrt], dim="time").sortby("time")

# Deduplicate if needed
times = pd.to_datetime(cams_all["time"].values)
if times.nunique() != len(times):
    cams_all = cams_all.sel(time=~cams_all.get_index("time").duplicated()).sortby("time")

# Align to SARAH time axis (reindex fills missing days with NaN)
cams_all = cams_all.sel(time=slice(sarah.time.values[0], sarah.time.values[-1]))
cams_all = cams_all.reindex(time=sarah["time"].values)

print("✅ CAMS aligned:", cams_all.sizes)
print("CAMS time range:", pd.to_datetime(cams_all.time.values).min(), "→", pd.to_datetime(cams_all.time.values).max())

✅ CAMS aligned: Frozen({'time': 457, 'lat': 20, 'lon': 20})
CAMS time range: 2024-01-01 00:00:00 → 2025-04-01 00:00:00


In [4]:
# =========================
# 2.3 Convert S2 monthly → daily
# =========================
# S2 months available: 2024 only
sarah_time = pd.to_datetime(sarah["time"].values)
sarah_month_labels = sarah_time.strftime("%Y-%m")

s2_month_index = pd.Index(s2_monthly["month"].values.astype(str))
band_names = ["ndvi", "brightness", "albedo", "nir"]

out_h = sarah.sizes["lat"]
out_w = sarah.sizes["lon"]
sarah_lats = sarah["lat"].values
sarah_lons = sarah["lon"].values

daily_s2_list = []
for m in sarah_month_labels:
    if m in s2_month_index:
        daily_s2_list.append(s2_monthly.sel(month=m).drop_vars("month"))
    else:
        # For 2025 months: fill NaNs
        empty = xr.Dataset(
            {k: (("lat", "lon"), np.full((out_h, out_w), np.nan, dtype="float32")) for k in band_names},
            coords={"lat": sarah_lats, "lon": sarah_lons}
        )
        daily_s2_list.append(empty)

s2_daily = xr.concat(daily_s2_list, dim="time").assign_coords(time=sarah["time"].values)

print("✅ S2 daily:", s2_daily.sizes)
print("Missing months example:", sorted(set([m for m in sarah_month_labels if m not in s2_month_index]))[:5])

✅ S2 daily: Frozen({'time': 457, 'lat': 20, 'lon': 20})
Missing months example: ['2025-01', '2025-02', '2025-03', '2025-04']


In [5]:
# =========================
# 2.4 Build fused Dataset
# =========================
# DEM: rename x/y -> lon/lat and broadcast to time
dem_ll = dem.rename({"x":"lon", "y":"lat"})
dem_time = dem_ll[["elevation","slope","aspect"]].expand_dims(time=sarah["time"].values)

def pick(ds, name):
    return ds[name] if (ds is not None and name in ds.data_vars) else None

fused_vars = {
    "SIS": sarah["SIS"],

    # CAMS (stable)
    "aod550": pick(cams_all, "aod550"),
    "tcwv":   pick(cams_all, "tcwv"),

    # optional if exists (often missing in NRT)
    "tcc":    pick(cams_all, "tcc"),

    # DEM (static)
    "elevation": dem_time["elevation"],
    "slope":     dem_time["slope"],
    "aspect":    dem_time["aspect"],

    # S2 daily (NaN in 2025)
    "ndvi":       s2_daily["ndvi"],
    "brightness": s2_daily["brightness"],
    "albedo":     s2_daily["albedo"],
    "nir":        s2_daily["nir"],
}
fused_vars = {k:v for k,v in fused_vars.items() if v is not None}

fused = xr.Dataset(fused_vars)

print("✅ fused:", fused.sizes)
print("vars:", list(fused.data_vars))

✅ fused: Frozen({'time': 457, 'lon': 20, 'lat': 20})
vars: ['SIS', 'aod550', 'tcwv', 'tcc', 'elevation', 'slope', 'aspect', 'ndvi', 'brightness', 'albedo', 'nir']


In [6]:
# =========================
# 2.5 Time features + I0 + Kt
# =========================
times = pd.to_datetime(fused.time.values)
doy = times.dayofyear.values.astype(int)

doy_sin = np.sin(2*np.pi*doy/365.25).astype("float32")
doy_cos = np.cos(2*np.pi*doy/365.25).astype("float32")

# Broadcast doy_sin/cos to (time, lat, lon)
H, W = fused.sizes["lat"], fused.sizes["lon"]
doy_sin_map = np.repeat(doy_sin[:, None, None], H, axis=1)
doy_sin_map = np.repeat(doy_sin_map, W, axis=2)
doy_cos_map = np.repeat(doy_cos[:, None, None], H, axis=1)
doy_cos_map = np.repeat(doy_cos_map, W, axis=2)

# Extraterrestrial radiation I0 (daily mean W/m²)
def extraterrestrial_radiation_Wm2(lat_deg, doy_arr):
    lat = np.deg2rad(lat_deg.astype(float))
    doy_arr = doy_arr.astype(float)

    dr = 1 + 0.033 * np.cos(2 * np.pi * doy_arr / 365.0)
    delta = 0.409 * np.sin(2 * np.pi * doy_arr / 365.0 - 1.39)
    ws = np.arccos(np.clip(-np.tan(lat) * np.tan(delta), -1, 1))
    Gsc = 1367.0

    Ra = (24 * 3600 / np.pi) * Gsc * dr * (
        ws * np.sin(lat) * np.sin(delta) +
        np.cos(lat) * np.cos(delta) * np.sin(ws)
    )
    return (Ra / (24 * 3600)).astype("float32")

lat_grid = fused["lat"].values.astype("float32")  # (H,)
I0_all = np.zeros((len(times), H, W), dtype="float32")
for i, d in enumerate(doy):
    I0_row = extraterrestrial_radiation_Wm2(lat_grid, np.full_like(lat_grid, d))
    I0_all[i, :, :] = np.repeat(I0_row[:, None], W, axis=1)

SIS_all = fused["SIS"].values.astype("float32")         # (T,H,W)
Kt_all  = (SIS_all / (I0_all + 1e-6)).astype("float32")
Kt_all  = np.clip(Kt_all, 0.0, 1.2)

print("✅ Built I0_all:", I0_all.shape, "| range:", float(I0_all.min()), "→", float(I0_all.max()))
print("✅ Built Kt_all:", Kt_all.shape, "| range:", float(Kt_all.min()), "→", float(Kt_all.max()))

# Save I0 + dates (used by UI + debug)
np.save(OUT_PRE / "dates_all.npy", times.strftime("%Y-%m-%d").values.astype("U10"))
np.save(OUT_PRE / "I0_all.npy", I0_all)
print("✅ Saved:", OUT_PRE / "dates_all.npy")
print("✅ Saved:", OUT_PRE / "I0_all.npy")

✅ Built I0_all: (457, 20, 20) | range: 72.9541244506836 → 483.22625732421875
✅ Built Kt_all: (457, 20, 20) | range: 0.06553531438112259 → 0.757155179977417
✅ Saved: C:\Users\Student\Desktop\SIS\data\precomputed\dates_all.npy
✅ Saved: C:\Users\Student\Desktop\SIS\data\precomputed\I0_all.npy


In [7]:
# =========================
# 2.6 Model A (2024 only)
# =========================
UNET_FEATURES_A = [
    # CAMS
    "tcc", "aod550", "tcwv",
    # DEM
    "elevation", "slope", "aspect",
    # S2
    "ndvi", "brightness", "albedo", "nir",
    # Seasonality
    "doy_sin", "doy_cos",
]

# Mask 2024
mask_A = (times >= pd.Timestamp("2024-01-01")) & (times <= pd.Timestamp("2024-12-31"))
idx_A = np.where(mask_A)[0]
dates_A = times[mask_A]

print("A days:", len(idx_A), "|", dates_A.min(), "→", dates_A.max())

# Build feature stack for A (T,H,W,C)
def get_var(name):
    if name == "doy_sin": return doy_sin_map
    if name == "doy_cos": return doy_cos_map
    return fused[name].values.astype("float32")

# Build X_A (all days, then slice)
X_A_all = np.stack([get_var(v) for v in UNET_FEATURES_A], axis=-1)  # (T,H,W,C)
X_A = X_A_all[idx_A]                                               # (TA,H,W,C)
yKt_A = Kt_all[idx_A][..., None]                                   # (TA,H,W,1)
I0_A = I0_all[idx_A]                                               # (TA,H,W)
SIS_true_A = SIS_all[idx_A]                                        # (TA,H,W)

print("✅ X_A:", X_A.shape, " yKt_A:", yKt_A.shape)

# Normalize using saved stats from training
train_mean_A = np.load(A_MEAN_PATH).astype("float32")
train_std_A  = np.load(A_STD_PATH).astype("float32")
X_A_norm = (X_A - train_mean_A) / (train_std_A + 1e-6)

# Predict with Model A
model_A = keras.models.load_model(MODEL_A_PATH)
Kt_pred_A = model_A.predict(X_A_norm, verbose=0).astype("float32")[..., 0]  # (TA,H,W)
Kt_pred_A = np.clip(Kt_pred_A, 0.0, 1.2)
SIS_pred_A = (Kt_pred_A * I0_A).astype("float32")                           # (TA,H,W)

city_mean_A = SIS_pred_A.mean(axis=(1,2)).astype("float32")
roi_pct_A = (SIS_pred_A / (city_mean_A[:, None, None] + 1e-6) * 100.0).astype("float32")

# Save A outputs for UI
np.save(OUT_PRE / "dates_A_2024.npy", dates_A.strftime("%Y-%m-%d").values.astype("U10"))
np.save(OUT_PRE / "SIS_pred_A_2024.npy", SIS_pred_A)
np.save(OUT_PRE / "city_mean_A_2024.npy", city_mean_A)
np.save(OUT_PRE / "roi_pct_A_2024.npy", roi_pct_A)

print("✅ Saved Model A UI assets in:", OUT_PRE)

A days: 366 | 2024-01-01 00:00:00 → 2024-12-31 00:00:00
✅ X_A: (366, 20, 20, 12)  yKt_A: (366, 20, 20, 1)
✅ Saved Model A UI assets in: C:\Users\Student\Desktop\SIS\data\precomputed


In [8]:
# =========================
# 2.7 Model B (future-safe)
# =========================
UNET_FEATURES_B = ["aod550","tcwv","elevation","slope","aspect","doy_sin","doy_cos"]

# UI support window for Model B
mask_B = (times >= pd.Timestamp("2024-01-01")) & (times <= pd.Timestamp("2025-03-31"))
idx_B = np.where(mask_B)[0]
dates_B = times[mask_B]

print("B days:", len(idx_B), "|", dates_B.min(), "→", dates_B.max())

def get_var_B(name):
    if name == "doy_sin": return doy_sin_map
    if name == "doy_cos": return doy_cos_map
    return fused[name].values.astype("float32")

X_B_all = np.stack([get_var_B(v) for v in UNET_FEATURES_B], axis=-1)  # (T,H,W,7)
X_B = X_B_all[idx_B]
I0_B = I0_all[idx_B]
SIS_true_B = SIS_all[idx_B]

print("✅ X_B:", X_B.shape)

# Normalize using saved stats from training
train_mean_B = np.load(B_MEAN_PATH).astype("float32")
train_std_B  = np.load(B_STD_PATH).astype("float32")
X_B_norm = (X_B - train_mean_B) / (train_std_B + 1e-6)

# Predict with Model B
model_B = keras.models.load_model(MODEL_B_PATH)
Kt_pred_B = model_B.predict(X_B_norm, verbose=0).astype("float32")[..., 0]  # (TB,H,W)
Kt_pred_B = np.clip(Kt_pred_B, 0.0, 1.2)
SIS_pred_B = (Kt_pred_B * I0_B).astype("float32")                           # (TB,H,W)

city_mean_B = SIS_pred_B.mean(axis=(1,2)).astype("float32")
roi_pct_B = (SIS_pred_B / (city_mean_B[:, None, None] + 1e-6) * 100.0).astype("float32")

# Save B outputs for UI
np.save(OUT_PRE / "dates_B.npy", dates_B.strftime("%Y-%m-%d").values.astype("U10"))
np.save(OUT_PRE / "SIS_pred_B.npy", SIS_pred_B)
np.save(OUT_PRE / "city_mean_B.npy", city_mean_B)
np.save(OUT_PRE / "roi_pct_B.npy", roi_pct_B)

print("✅ Saved Model B UI assets in:", OUT_PRE)

B days: 456 | 2024-01-01 00:00:00 → 2025-03-31 00:00:00
✅ X_B: (456, 20, 20, 7)
✅ Saved Model B UI assets in: C:\Users\Student\Desktop\SIS\data\precomputed


In [9]:
# =========================
# 2.8 Terrain explain layers
# =========================
# DEM is static in fused already, take first day slice
elev = fused["elevation"].values.astype("float32")[0]
slope = fused["slope"].values.astype("float32")[0]
aspect = fused["aspect"].values.astype("float32")[0]

np.save(OUT_EXP / "elevation.npy", elev)
np.save(OUT_EXP / "slope.npy", slope)
np.save(OUT_EXP / "aspect.npy", aspect)

print("✅ Saved terrain layers:", OUT_EXP)

✅ Saved terrain layers: C:\Users\Student\Desktop\SIS\data\explain


In [10]:
# =========================
# 2.9 NDVI monthly explain layer
# =========================
months = s2_monthly["month"].values.astype(str)
ndvi_m = s2_monthly["ndvi"].values.astype("float32")   # (12,20,20)

np.save(OUT_EXP / "months_2024.npy", months.astype("U7"))
np.save(OUT_EXP / "ndvi_monthly_2024.npy", ndvi_m)

print("✅ Saved NDVI monthly:", OUT_EXP)
print("months:", list(months))

✅ Saved NDVI monthly: C:\Users\Student\Desktop\SIS\data\explain
months: [np.str_('2024-01'), np.str_('2024-02'), np.str_('2024-03'), np.str_('2024-04'), np.str_('2024-05'), np.str_('2024-06'), np.str_('2024-07'), np.str_('2024-08'), np.str_('2024-09'), np.str_('2024-10'), np.str_('2024-11'), np.str_('2024-12')]


In [11]:
# =========================
# 2.10 Cloud daily explain layer
# =========================
if "tcc" in fused.data_vars:
    cloud = fused["tcc"].values.astype("float32")  # (T,H,W) maybe NaNs in NRT
    # Align to Model B supported window (same dates_B)
    cloud_B = cloud[idx_B]                         # (TB,H,W)
    np.save(OUT_EXP / "cloud_daily_B.npy", cloud_B)
    print("✅ Saved cloud daily (aligned to dates_B):", OUT_EXP / "cloud_daily_B.npy")
    print("Cloud NaN ratio:", float(np.isnan(cloud_B).sum()) / cloud_B.size)
else:
    print("⚠ No `tcc` in fused → cloud layer not available. (UI should hide toggle or show warning.)")

✅ Saved cloud daily (aligned to dates_B): C:\Users\Student\Desktop\SIS\data\explain\cloud_daily_B.npy
Cloud NaN ratio: 0.19736842105263158


In [12]:
# =========================
# 2.11 Save UI meta
# =========================
ui_meta = {
    "mode_switch_rule": "Use Model A for 2024 dates, else Model B",
    "modelA": {
        "dates_file": "data/precomputed/dates_A_2024.npy",
        "sis_file": "data/precomputed/SIS_pred_A_2024.npy",
        "mean_file": "data/precomputed/city_mean_A_2024.npy",
        "roi_pct_file": "data/precomputed/roi_pct_A_2024.npy",
    },
    "modelB": {
        "dates_file": "data/precomputed/dates_B.npy",
        "sis_file": "data/precomputed/SIS_pred_B.npy",
        "mean_file": "data/precomputed/city_mean_B.npy",
        "roi_pct_file": "data/precomputed/roi_pct_B.npy",
    },
    "explain": {
        "terrain": {
            "elevation": "data/explain/elevation.npy",
            "slope": "data/explain/slope.npy",
            "aspect": "data/explain/aspect.npy"
        },
        "ndvi_monthly": {
            "months": "data/explain/months_2024.npy",
            "ndvi": "data/explain/ndvi_monthly_2024.npy",
            "note": "For 2025, reuse same month from 2024 as a demo explanation layer."
        },
        "cloud_daily": {
            "cloud_B": "data/explain/cloud_daily_B.npy",
            "note": "May contain NaNs in CAMS NRT period if tcc missing."
        }
    }
}

meta_path = OUT_PRE / "ui_assets_meta.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(ui_meta, f, indent=2)

print("✅ Saved:", meta_path)

✅ Saved: C:\Users\Student\Desktop\SIS\data\precomputed\ui_assets_meta.json


In [2]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

BASE_DIR = Path("data")
SARAH_MERGED_FILE = BASE_DIR / "SARAH3" / "sarah3_sis_eindhoven_0.05_2024-01-01_2025-04-01.nc"

OUT_DIR = BASE_DIR / "precomputed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ds = xr.open_dataset(SARAH_MERGED_FILE)

truth_mean = ds["SIS"].mean(dim=("lat", "lon")).values.astype("float32")
dates = pd.to_datetime(ds["time"].values).normalize()

np.save(OUT_DIR / "sarah_city_mean_all.npy", truth_mean)
np.save(OUT_DIR / "sarah_dates_all.npy", dates.values.astype("datetime64[ns]"))

print("✅ Saved truth mean:", OUT_DIR / "sarah_city_mean_all.npy")
print("✅ Saved dates     :", OUT_DIR / "sarah_dates_all.npy")
print("Days:", len(truth_mean),
      "Range:", float(np.nanmin(truth_mean)), "→", float(np.nanmax(truth_mean)))

✅ Saved truth mean: data\precomputed\sarah_city_mean_all.npy
✅ Saved dates     : data\precomputed\sarah_dates_all.npy
Days: 457 Range: 7.284999847412109 → 350.7025146484375
